# Transfer learning: DenseNet121 and ResNet50

Train and fine-tune ImageNet-pretrained models on the cleaned `chest_xray_cnn_splits` folders produced by `chest_xray_train_val_test_split.ipynb`. Run that notebook first. Validation and test images remain unaugmented; test results are reported only after training.


In [ ]:
from pathlib import Path
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
HEAD_EPOCHS = 5
FINE_TUNE_EPOCHS = 10
FINE_TUNE_LAYERS = 30
AUTOTUNE = tf.data.AUTOTUNE

tf.keras.utils.set_random_seed(SEED)


## Locate the cleaned data and load datasets

If automatic discovery finds multiple split folders, set `SPLIT_DIR` explicitly to the intended folder.


In [ ]:
SPLIT_DIR = None  # Example: Path(r"C:/.../chest_xray_cnn_splits")
if SPLIT_DIR is None:
    candidates = [Path.cwd() / "chest_xray_cnn_splits"]
    for root in (Path.home() / ".cache/kagglehub", Path.cwd()):
        if root.is_dir():
            candidates.extend(root.rglob("chest_xray_cnn_splits"))
    SPLIT_DIR = next((p for p in candidates if all((p / s).is_dir() for s in ("train", "val", "test"))), None)
if SPLIT_DIR is None or not all((Path(SPLIT_DIR) / s).is_dir() for s in ("train", "val", "test")):
    raise FileNotFoundError("Set SPLIT_DIR to the cleaned chest_xray_cnn_splits folder created by the split notebook.")
SPLIT_DIR = Path(SPLIT_DIR)

train_ds = tf.keras.utils.image_dataset_from_directory(SPLIT_DIR / "train", image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, seed=SEED, label_mode="int")
class_names = train_ds.class_names
val_ds = tf.keras.utils.image_dataset_from_directory(SPLIT_DIR / "val", image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=False, class_names=class_names, label_mode="int")
test_ds = tf.keras.utils.image_dataset_from_directory(SPLIT_DIR / "test", image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=False, class_names=class_names, label_mode="int")
if len(class_names) != 2:
    raise ValueError(f"Expected two classes, found {class_names}")
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)
print("Data:", SPLIT_DIR, "| class order:", class_names)


## Training setup

Use the project imbalance notebook's class-weighted approach in spirit: weights come only from training labels. Each model first learns its new classifier head, then unfreezes the final backbone layers at a lower learning rate. The augmentation is applied only while fitting.


In [ ]:
# Count labels from training batches only and inverse-frequency weight them.
counts = np.zeros(len(class_names), dtype=np.int64)
for _, labels in train_ds:
    counts += np.bincount(labels.numpy().reshape(-1), minlength=len(class_names))
if np.any(counts == 0):
    raise ValueError(f"Training split is missing a class: {dict(zip(class_names, counts.tolist()))}")
class_weight = {i: counts.sum() / (len(class_names) * count) for i, count in enumerate(counts)}
print("Training counts:", dict(zip(class_names, counts.tolist())))
print("Class weights:", class_weight)

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal", seed=SEED),
    tf.keras.layers.RandomRotation(0.02, seed=SEED + 1),
    tf.keras.layers.RandomZoom(0.05, seed=SEED + 2),
], name="training_augmentation")


## Build, train, predict, and evaluate both models

Models use their matching Keras preprocessing function and a two-class softmax output. Evaluation includes Keras loss/accuracy, a classification report, and a confusion matrix on the held-out test split.


In [ ]:
def make_model(architecture):
    if architecture == "DenseNet121":
        backbone = tf.keras.applications.DenseNet121(include_top=False, weights="imagenet", input_shape=(*IMAGE_SIZE, 3))
        preprocess = tf.keras.applications.densenet.preprocess_input
    elif architecture == "ResNet50":
        backbone = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=(*IMAGE_SIZE, 3))
        preprocess = tf.keras.applications.resnet50.preprocess_input
    else:
        raise ValueError(f"Unsupported architecture: {architecture}")
    backbone.trainable = False
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
    x = augmentation(inputs)
    x = tf.keras.layers.Lambda(preprocess, name="preprocess")(x)
    x = backbone(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs, name=architecture.lower())
    return model, backbone


def train_model(architecture):
    model, backbone = make_model(architecture)
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=1, min_lr=1e-7),
    ]
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    head_history = model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, class_weight=class_weight, callbacks=callbacks)

    backbone.trainable = True
    for layer in backbone.layers[:-FINE_TUNE_LAYERS]:
        layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    fine_history = model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS, class_weight=class_weight, callbacks=callbacks)
    return model, {"head": head_history, "fine_tune": fine_history}


results = {name: train_model(name) for name in ("DenseNet121", "ResNet50")}

# Save both final models before threshold selection or test evaluation.
MODEL_DIR = Path.cwd() / "saved_models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
for name, (model, _) in results.items():
    model_path = MODEL_DIR / f"{name.lower()}_transfer.keras"
    model.save(model_path)
    print(f"Saved {name} to {model_path}")

# Select the DenseNet121 threshold by pneumonia F1 on validation data only.
thresholds = (0.3, 0.4, 0.5, 0.6, 0.7)
positive_index = class_names.index("PNEUMONIA")
dense_model = results["DenseNet121"][0]
val_true, val_positive_probabilities = [], []
for images, labels in val_ds:
    val_true.extend(labels.numpy().reshape(-1))
    val_positive_probabilities.extend(dense_model(images, training=False).numpy()[:, positive_index])
val_true = np.asarray(val_true)
val_positive_probabilities = np.asarray(val_positive_probabilities)

threshold_results = []
for threshold in thresholds:
    val_pred = np.where(val_positive_probabilities >= threshold, positive_index, 1 - positive_index)
    precision, recall, f1, _ = precision_recall_fscore_support(
        val_true, val_pred, average="binary", pos_label=positive_index, zero_division=0
    )
    threshold_results.append({"threshold": threshold, "precision": precision, "recall": recall, "f1": f1})
    print(f"Validation threshold {threshold:.1f} | precision: {precision:.4f} | recall: {recall:.4f} | F1: {f1:.4f}")

selected_threshold = max(threshold_results, key=lambda row: row["f1"])["threshold"]
print(f"Selected threshold by validation F1: {selected_threshold:.1f}")

# Evaluate the selected threshold once on the held-out test split.
test_true, test_positive_probabilities = [], []
for images, labels in test_ds:
    test_true.extend(labels.numpy().reshape(-1))
    test_positive_probabilities.extend(dense_model(images, training=False).numpy()[:, positive_index])
test_true = np.asarray(test_true)
test_positive_probabilities = np.asarray(test_positive_probabilities)
test_pred = np.where(test_positive_probabilities >= selected_threshold, positive_index, 1 - positive_index)
print(f"\nDenseNet121 test results at threshold {selected_threshold:.1f}:")
print(classification_report(test_true, test_pred, target_names=class_names, digits=4, zero_division=0))
print("Confusion matrix (rows=true, columns=predicted):\n", confusion_matrix(test_true, test_pred, labels=range(len(class_names))))


## Compare training curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, result in results.items():
    history = result[1]
    for metric, ax in (("accuracy", axes[0]), ("loss", axes[1])):
        values = history["head"].history[metric] + history["fine_tune"].history[metric]
        ax.plot(values, label=name)
        ax.set_title(metric.capitalize())
        ax.set_xlabel("Epoch")
        ax.legend()
plt.tight_layout()
plt.show()
